# 📊 RIVA — School Health Analytics
**الهدف:** تحليل بيانات الطلاب + رسم داشبورد كامل

- توزيع BMI للفصل
- Anomaly Detection
- مقارنة بمعايير WHO
- تقرير جاهز للمدرسة

## 1️⃣ تثبيت

In [ ]:
!pip install matplotlib seaborn plotly -q
print('✅ جاهز')

## 2️⃣ رفع الملفات

In [ ]:
from google.colab import files
print('ارفع: who_growth_standards.json + cluster_centers.json')
uploaded = files.upload()
print('✅', list(uploaded.keys()))

## 3️⃣ بيانات تجريبية للفصل

In [ ]:
import json, numpy as np, pandas as pd

with open('who_growth_standards.json', encoding='utf-8') as f:
    who_data = json.load(f)
with open('cluster_centers.json', encoding='utf-8') as f:
    cluster_data = json.load(f)

# بيانات 30 طالب تجريبية
np.random.seed(42)
students = []
for i in range(30):
    gender = np.random.choice(['boys','girls'])
    age_m  = np.random.randint(84, 156)
    height = np.random.normal(130, 12)
    weight = np.random.normal(28, 8)
    weight = max(15, weight)
    students.append({
        'name':       f'طالب {i+1}',
        'gender':     gender,
        'age_months': age_m,
        'height_cm':  round(height, 1),
        'weight_kg':  round(weight, 1)
    })

print(f'✅ {len(students)} طالب جاهزين للتحليل')

## 4️⃣ تحليل كل طالب

In [ ]:
def z_score(value, L, M, S):
    if L != 0: return (((value/M)**L)-1)/(L*S)
    return np.log(value/M)/S

centers = np.array([[c['age_months'],c['gender'],c['bmi'],c['z_score']]
                     for c in cluster_data['centers']])
clabels = {i: c['label'] for i,c in enumerate(cluster_data['centers'])}

results = []
for s in students:
    bmi = round(s['weight_kg'] / ((s['height_cm']/100)**2), 2)
    table = who_data[f"bmi_{s['gender']}"]
    row = next((r for r in table if r['months']==s['age_months']), None)
    if not row: continue
    z = round(z_score(bmi, row['L'], row['M'], row['S']), 2)
    if   z >  2: status, color = 'وزن زائد', 'red'
    elif z < -2: status, color = 'نحافة', 'orange'
    else:        status, color = 'طبيعي', 'green'
    g_num = 1 if s['gender']=='boys' else 0
    point = np.array([s['age_months'], g_num, bmi, z])
    dists = [np.linalg.norm(point-c) for c in centers]
    nearest = int(np.argmin(dists))
    is_anomaly = min(dists) > 3.0
    results.append({**s, 'bmi':bmi, 'z_score':z, 'status':status,
                    'color':color, 'cluster':clabels[nearest],
                    'is_anomaly':is_anomaly})

df = pd.DataFrame(results)
print(f'✅ تم تحليل {len(df)} طالب')
print(df['status'].value_counts())

## 5️⃣ داشبورد الفصل

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 تقرير الصحة المدرسية — ريفا', fontsize=16, fontweight='bold')

# 1. توزيع الحالات
counts = df['status'].value_counts()
colors_pie = ['#2ecc71' if s=='طبيعي' else '#e74c3c' if s=='وزن زائد' else '#f39c12'
              for s in counts.index]
axes[0,0].pie(counts.values, labels=counts.index, colors=colors_pie,
              autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('توزيع حالات الطلاب')

# 2. توزيع BMI
color_map = {'طبيعي':'#2ecc71','وزن زائد':'#e74c3c','نحافة':'#f39c12'}
for status, group in df.groupby('status'):
    axes[0,1].hist(group['bmi'], alpha=0.7, label=status,
                   color=color_map.get(status,'gray'), bins=10)
axes[0,1].set_title('توزيع BMI')
axes[0,1].set_xlabel('BMI')
axes[0,1].legend()

# 3. Z-Score لكل طالب
colors_bar = [color_map.get(s,'gray') for s in df['status']]
axes[1,0].bar(range(len(df)), df['z_score'], color=colors_bar, alpha=0.8)
axes[1,0].axhline(y=2,  color='red',    linestyle='--', alpha=0.5, label='+2 SD')
axes[1,0].axhline(y=-2, color='orange', linestyle='--', alpha=0.5, label='-2 SD')
axes[1,0].set_title('Z-Score لكل طالب')
axes[1,0].set_xlabel('رقم الطالب')
axes[1,0].legend()

# 4. Anomalies
anomalies = df[df['is_anomaly']==True]
normal    = df[df['is_anomaly']==False]
axes[1,1].scatter(normal['age_months'],    normal['z_score'],
                  c='#2ecc71', alpha=0.7, label=f'طبيعي ({len(normal)})', s=60)
axes[1,1].scatter(anomalies['age_months'], anomalies['z_score'],
                  c='#e74c3c', marker='*', s=200,
                  label=f'حالة طارئة ({len(anomalies)})')
axes[1,1].set_title('Anomaly Detection')
axes[1,1].set_xlabel('العمر (شهر)')
axes[1,1].set_ylabel('Z-Score')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('school_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ الداشبورد جاهز')

## 6️⃣ تقرير نصي للمدرسة

In [ ]:
total    = len(df)
normal   = len(df[df['status']=='طبيعي'])
overweight = len(df[df['status']=='وزن زائد'])
thin     = len(df[df['status']=='نحافة'])
anomaly  = len(df[df['is_anomaly']==True])
avg_z    = round(df['z_score'].mean(), 2)

if   avg_z >  1: trend = 'يميل للوزن الزائد ⚠️'
elif avg_z < -1: trend = 'يميل للنحافة ⚠️'
else:            trend = 'توزيع طبيعي ✅'

print('='*45)
print('📋 تقرير الصحة المدرسية — ريفا')
print('='*45)
print(f'إجمالي الطلاب:    {total}')
print(f'طبيعي:            {normal} ({round(normal/total*100)}%)')
print(f'وزن زائد/سمنة:   {overweight} ({round(overweight/total*100)}%)')
print(f'نحافة:            {thin} ({round(thin/total*100)}%)')
print(f'حالات طارئة:     {anomaly}')
print(f'متوسط Z-Score:   {avg_z}')
print(f'اتجاه الفصل:     {trend}')
print('='*45)

if anomaly > 0:
    print('\n⚠️ الحالات الطارئة:')
    for _, row in df[df['is_anomaly']==True].iterrows():
        print(f'  - {row["name"]}: BMI={row["bmi"]} | Z={row["z_score"]} | {row["status"]}')

## 7️⃣ تحميل الداشبورد

In [ ]:
from google.colab import files
files.download('school_dashboard.png')
print('✅ احفظه في: business-intelligence/pitch/')